#  Notebook 1: Stock Price Prediction
**Dataset:** Multi-Stock Price Dataset (5 Stocks, 365 Days)

**Goal:** Predict next-day stock closing price using ensemble ML + technical indicators

**Why NOT LSTM?** Only 365 data points -- far too few for deep learning. Traditional ML models
(XGBoost, Random Forest, Gradient Boosting) excel with small tabular datasets.

**Pipeline:**
1. Load & explore the dataset
2. Feature engineering (Technical indicators + lag features)
3. Train XGBoost, Random Forest, Gradient Boosting
4. Hyperparameter tuning via GridSearchCV
5. Weighted ensemble for best R²
6. Predict next 30 days & visualize

## Step 1: Install Dependencies & Upload Dataset

In [ ]:
# Install required libraries
!pip install scikit-learn xgboost lightgbm pandas numpy matplotlib seaborn -q

# === FOR GOOGLE COLAB: Upload your stock_data.csv ===
# Uncomment the next 3 lines if running in Google Colab
# from google.colab import files
# print("Upload your stock_data.csv file")
# uploaded = files.upload()

print(" Dependencies ready!")

## Step 2: Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit, cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import (RandomForestRegressor, GradientBoostingRegressor,
                               ExtraTreesRegressor, AdaBoostRegressor)
from sklearn.linear_model import Ridge, Lasso, ElasticNet

try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print(" XGBoost not installed")

try:
    from lightgbm import LGBMRegressor
    HAS_LGBM = True
except ImportError:
    HAS_LGBM = False
    print(" LightGBM not installed")

import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)

print(" Libraries imported!")

## Step 3: Load & Explore Data

In [ ]:
import os

# Try multiple paths
possible_paths = ['stock_data.csv', '/content/stock_data.csv', 'stock_data/stock_data.csv']
df = None
for p in possible_paths:
    if os.path.exists(p):
        df = pd.read_csv(p)
        print(f" Loaded from: {p}")
        break

if df is None:
    raise FileNotFoundError("stock_data.csv not found! Please upload it.")

print(f"\nShape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head(10)

In [ ]:
# Basic info
print("Dataset Info:")
print(df.dtypes)
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nStatistics:")
df.describe()

In [ ]:
# Rename the date column and parse dates
date_col = df.columns[0]  # 'Unnamed: 0' is the date column
df = df.rename(columns={date_col: 'Date'})
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)

# Get all stock columns
stock_cols = [c for c in df.columns if 'Stock' in c]
print(f"Stock columns: {stock_cols}")
print(f"Date range: {df['Date'].min()} to {df['Date'].max()}")
print(f"Number of trading days: {len(df)}")

# Visualize all stocks
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Multi-Stock Price History -- Exploratory Data Analysis', fontsize=16, fontweight='bold')

colors = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63', '#9C27B0']
for i, col in enumerate(stock_cols):
    ax = axes[i // 3, i % 3]
    ax.plot(df['Date'], df[col], color=colors[i], linewidth=1.5, label=col)
    ax.set_title(f'{col} Price Over Time', fontsize=12)
    ax.set_xlabel('Date')
    ax.set_ylabel('Price ($)')
    ax.grid(alpha=0.3)
    ax.legend()

# Correlation heatmap
ax = axes[1, 2]
corr = df[stock_cols].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, ax=ax, fmt='.2f')
ax.set_title('Stock Correlation Matrix', fontsize=12)

plt.tight_layout()
plt.savefig('stock_eda.png', dpi=150, bbox_inches='tight')
plt.show()
print(" EDA complete!")

## Step 4: Feature Engineering (Lag Features + Technical Indicators)

In [ ]:
TARGET = 'Stock_1'
print(f"Target: Predict next-day price for '{TARGET}'")

feat_df = df.copy()

# ========== LAG FEATURES (most important for time series with ML) ==========
for lag in [1, 2, 3, 5, 7, 10, 14]:
    feat_df[f'{TARGET}_lag{lag}'] = feat_df[TARGET].shift(lag)

# Lag features for other stocks (cross-stock signal)
for col in stock_cols:
    if col != TARGET:
        feat_df[f'{col}_lag1'] = feat_df[col].shift(1)
        feat_df[f'{col}_lag3'] = feat_df[col].shift(3)

# ========== MOVING AVERAGES ==========
for window in [3, 5, 7, 10, 14, 21]:
    feat_df[f'{TARGET}_MA{window}'] = feat_df[TARGET].shift(1).rolling(window).mean()
    feat_df[f'{TARGET}_std{window}'] = feat_df[TARGET].shift(1).rolling(window).std()

# ========== EMA ==========
for span in [5, 10, 21]:
    feat_df[f'{TARGET}_EMA{span}'] = feat_df[TARGET].shift(1).ewm(span=span, adjust=False).mean()

# ========== RSI ==========
delta = feat_df[TARGET].diff()
gain = delta.where(delta > 0, 0).rolling(14).mean()
loss_val = (-delta.where(delta < 0, 0)).rolling(14).mean()
rs = gain / (loss_val + 1e-10)
feat_df[f'{TARGET}_RSI'] = 100 - (100 / (1 + rs))
feat_df[f'{TARGET}_RSI'] = feat_df[f'{TARGET}_RSI'].shift(1)

# ========== BOLLINGER BANDS ==========
ma20 = feat_df[TARGET].shift(1).rolling(20).mean()
std20 = feat_df[TARGET].shift(1).rolling(20).std()
feat_df[f'{TARGET}_BB_upper'] = ma20 + 2 * std20
feat_df[f'{TARGET}_BB_lower'] = ma20 - 2 * std20
feat_df[f'{TARGET}_BB_width'] = feat_df[f'{TARGET}_BB_upper'] - feat_df[f'{TARGET}_BB_lower']
feat_df[f'{TARGET}_BB_pct'] = (feat_df[TARGET].shift(1) - feat_df[f'{TARGET}_BB_lower']) / (feat_df[f'{TARGET}_BB_width'] + 1e-10)

# ========== MACD ==========
ema12 = feat_df[TARGET].shift(1).ewm(span=12, adjust=False).mean()
ema26 = feat_df[TARGET].shift(1).ewm(span=26, adjust=False).mean()
feat_df[f'{TARGET}_MACD'] = ema12 - ema26
feat_df[f'{TARGET}_MACD_signal'] = feat_df[f'{TARGET}_MACD'].ewm(span=9, adjust=False).mean()
feat_df[f'{TARGET}_MACD_hist'] = feat_df[f'{TARGET}_MACD'] - feat_df[f'{TARGET}_MACD_signal']

# ========== PRICE CHANGES / MOMENTUM ==========
for period in [1, 2, 3, 5, 7, 10]:
    feat_df[f'{TARGET}_return_{period}d'] = feat_df[TARGET].pct_change(period).shift(1)

# ========== PRICE RATIOS ==========
feat_df[f'{TARGET}_price_to_MA5'] = feat_df[TARGET].shift(1) / (feat_df[f'{TARGET}_MA5'] + 1e-10)
feat_df[f'{TARGET}_price_to_MA10'] = feat_df[TARGET].shift(1) / (feat_df[f'{TARGET}_MA10'] + 1e-10)
feat_df[f'{TARGET}_price_to_MA21'] = feat_df[TARGET].shift(1) / (feat_df[f'{TARGET}_MA21'] + 1e-10)

# ========== TEMPORAL FEATURES ==========
feat_df['day_of_week'] = feat_df['Date'].dt.dayofweek
feat_df['day_of_month'] = feat_df['Date'].dt.day
feat_df['month'] = feat_df['Date'].dt.month
feat_df['day_of_year'] = feat_df['Date'].dt.dayofyear
feat_df['sin_doy'] = np.sin(2 * np.pi * feat_df['day_of_year'] / 365)
feat_df['cos_doy'] = np.cos(2 * np.pi * feat_df['day_of_year'] / 365)

# ========== INTER-STOCK FEATURES ==========
feat_df['stock_mean'] = df[stock_cols].shift(1).mean(axis=1)
feat_df['stock_std'] = df[stock_cols].shift(1).std(axis=1)
feat_df[f'{TARGET}_vs_mean'] = df[TARGET].shift(1) - feat_df['stock_mean']

# ========== DROP NaN rows (from lag/rolling) ==========
feat_df = feat_df.dropna().reset_index(drop=True)

print(f"\nDataset after feature engineering: {feat_df.shape}")
feature_cols = [c for c in feat_df.columns if c not in ['Date', TARGET]]
print(f"Total features: {len(feature_cols)}")
print(f"\nSample features: {feature_cols[:15]}...")

## Step 5: Prepare Train/Test Split (Time-Based)

In [ ]:
X = feat_df[feature_cols].values
y = feat_df[TARGET].values
dates = feat_df['Date'].values

# Time-based split -- last 20% for testing (DO NOT shuffle for time series!)
split_idx = int(0.80 * len(X))

X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]
dates_train, dates_test = dates[:split_idx], dates[split_idx:]

# Scale features
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print(f"Train: {X_train_s.shape} | Test: {X_test_s.shape}")
print(f"Train period: {pd.Timestamp(dates_train[0]).date()} to {pd.Timestamp(dates_train[-1]).date()}")
print(f"Test period:  {pd.Timestamp(dates_test[0]).date()} to {pd.Timestamp(dates_test[-1]).date()}")
print(f"\nTarget stats (train): mean={y_train.mean():.2f}, std={y_train.std():.2f}")
print(f"Target stats (test):  mean={y_test.mean():.2f}, std={y_test.std():.2f}")

## Step 6: Train Multiple Models

In [ ]:
# Define models
models = {}

models['Ridge'] = Ridge(alpha=1.0)

models['Random Forest'] = RandomForestRegressor(
    n_estimators=500, max_depth=10, min_samples_split=5,
    min_samples_leaf=2, max_features='sqrt', random_state=42, n_jobs=-1
)

models['Extra Trees'] = ExtraTreesRegressor(
    n_estimators=500, max_depth=10, min_samples_split=5,
    min_samples_leaf=2, random_state=42, n_jobs=-1
)

models['Gradient Boosting'] = GradientBoostingRegressor(
    n_estimators=500, max_depth=4, learning_rate=0.05,
    subsample=0.8, min_samples_split=5, random_state=42
)

if HAS_XGB:
    models['XGBoost'] = XGBRegressor(
        n_estimators=500, max_depth=5, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1,
        reg_lambda=1.0, random_state=42, n_jobs=-1
    )

if HAS_LGBM:
    models['LightGBM'] = LGBMRegressor(
        n_estimators=500, max_depth=5, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1,
        reg_lambda=1.0, random_state=42, n_jobs=-1, verbose=-1
    )

# Train and evaluate
results = {}
tscv = TimeSeriesSplit(n_splits=5)

print(f"{'Model':<22} {'CV R²':<18} {'Test R²':<10} {'Test RMSE':<12} {'Test MAE':<10}")
print("=" * 72)

for name, model in models.items():
    # Time-series cross validation on training data
    cv_scores = cross_val_score(model, X_train_s, y_train, cv=tscv, scoring='r2')
    
    # Fit on full training data
    model.fit(X_train_s, y_train)
    
    # Predict
    y_pred = model.predict(X_test_s)
    
    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    
    results[name] = {
        'model': model,
        'cv_r2': cv_scores.mean(),
        'cv_std': cv_scores.std(),
        'r2': r2,
        'rmse': rmse,
        'mae': mae,
        'y_pred': y_pred
    }
    
    print(f"{name:<22} {cv_scores.mean():.4f}±{cv_scores.std():.4f}   {r2:.4f}     {rmse:.4f}       {mae:.4f}")

print("\n All models trained!")

## Step 7: Hyperparameter Tuning (Top Models)

In [ ]:
print(" Hyperparameter tuning with TimeSeriesSplit...\n")

# Tune Gradient Boosting
gb_params = {
    'n_estimators': [300, 500, 800],
    'max_depth': [3, 4, 5, 6],
    'learning_rate': [0.01, 0.03, 0.05, 0.1],
    'subsample': [0.7, 0.8, 0.9],
    'min_samples_split': [3, 5, 10]
}

gb_grid = GridSearchCV(
    GradientBoostingRegressor(random_state=42),
    gb_params, cv=tscv, scoring='r2', n_jobs=-1, verbose=0
)
gb_grid.fit(X_train_s, y_train)
print(f"Best GB: R²={gb_grid.best_score_:.4f}")
print(f"  Params: {gb_grid.best_params_}")

# Tune XGBoost
if HAS_XGB:
    xgb_params = {
        'n_estimators': [300, 500, 800],
        'max_depth': [3, 4, 5, 6],
        'learning_rate': [0.01, 0.03, 0.05, 0.1],
        'subsample': [0.7, 0.8, 0.9],
        'colsample_bytree': [0.7, 0.8, 0.9]
    }
    
    xgb_grid = GridSearchCV(
        XGBRegressor(random_state=42, n_jobs=-1),
        xgb_params, cv=tscv, scoring='r2', n_jobs=-1, verbose=0
    )
    xgb_grid.fit(X_train_s, y_train)
    print(f"\nBest XGB: R²={xgb_grid.best_score_:.4f}")
    print(f"  Params: {xgb_grid.best_params_}")

# Tune LightGBM
if HAS_LGBM:
    lgbm_params = {
        'n_estimators': [300, 500, 800],
        'max_depth': [3, 4, 5, 6, -1],
        'learning_rate': [0.01, 0.03, 0.05, 0.1],
        'subsample': [0.7, 0.8, 0.9],
        'colsample_bytree': [0.7, 0.8, 0.9]
    }
    
    lgbm_grid = GridSearchCV(
        LGBMRegressor(random_state=42, n_jobs=-1, verbose=-1),
        lgbm_params, cv=tscv, scoring='r2', n_jobs=-1, verbose=0
    )
    lgbm_grid.fit(X_train_s, y_train)
    print(f"\nBest LGBM: R²={lgbm_grid.best_score_:.4f}")
    print(f"  Params: {lgbm_grid.best_params_}")

print("\n Tuning complete!")

## Step 8: Weighted Ensemble

In [ ]:
# Collect tuned models
tuned_models = {}
tuned_models['Gradient Boosting (Tuned)'] = gb_grid.best_estimator_

if HAS_XGB:
    tuned_models['XGBoost (Tuned)'] = xgb_grid.best_estimator_
if HAS_LGBM:
    tuned_models['LightGBM (Tuned)'] = lgbm_grid.best_estimator_

# Add some base models too
tuned_models['Extra Trees'] = results['Extra Trees']['model']
tuned_models['Random Forest'] = results['Random Forest']['model']

# Evaluate each tuned model on test set
tuned_results = {}
print(f"{'Model':<30} {'Test R²':<10} {'Test RMSE':<12} {'Test MAE':<10}")
print("=" * 62)

for name, model in tuned_models.items():
    y_pred = model.predict(X_test_s)
    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    
    tuned_results[name] = {
        'model': model,
        'r2': r2,
        'rmse': rmse,
        'mae': mae,
        'y_pred': y_pred
    }
    print(f"{name:<30} {r2:.4f}     {rmse:.4f}       {mae:.4f}")

# ===== WEIGHTED ENSEMBLE =====
# Weight by R² score (better models get more weight)
weights = {}
for name, res in tuned_results.items():
    weights[name] = max(0.01, res['r2'])  # floor at 0.01

total_w = sum(weights.values())
weights = {k: v / total_w for k, v in weights.items()}

print(f"\nEnsemble weights:")
for name, w in weights.items():
    print(f"  {name}: {w:.3f}")

# Weighted average prediction
y_pred_ensemble = np.zeros_like(y_test, dtype=float)
for name, res in tuned_results.items():
    y_pred_ensemble += weights[name] * res['y_pred']

# Ensemble metrics
r2_ens = r2_score(y_test, y_pred_ensemble)
rmse_ens = np.sqrt(mean_squared_error(y_test, y_pred_ensemble))
mae_ens = mean_absolute_error(y_test, y_pred_ensemble)
mape_ens = np.mean(np.abs((y_test - y_pred_ensemble) / (y_test + 1e-8))) * 100

# Directional accuracy
actual_dir = np.diff(y_test) > 0
pred_dir = np.diff(y_pred_ensemble) > 0
dir_acc = np.mean(actual_dir == pred_dir) * 100

print("\n" + "=" * 55)
print("     ENSEMBLE -- FINAL EVALUATION")
print("=" * 55)
print(f"  R² Score          : {r2_ens:.4f}")
print(f"  RMSE              : ${rmse_ens:.4f}")
print(f"  MAE               : ${mae_ens:.4f}")
print(f"  MAPE              : {mape_ens:.2f}%")
print(f"  Directional Acc   : {dir_acc:.2f}%")
print("=" * 55)

if r2_ens >= 0.85:
    print(f"\n  TARGET ACHIEVED! R² = {r2_ens:.4f} ≥ 0.85")
elif r2_ens >= 0.75:
    print(f"\n Good result! R² = {r2_ens:.4f}")
else:
    print(f"\n R² = {r2_ens:.4f}")

## Step 9: Visualization -- Results & Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
fig.suptitle('Stock Price Prediction -- Model Evaluation', fontsize=16, fontweight='bold')

# 1. Model R² comparison
ax = axes[0, 0]
all_names = list(tuned_results.keys()) + ['ENSEMBLE']
all_r2 = [tuned_results[m]['r2'] for m in tuned_results.keys()] + [r2_ens]
bar_colors = ['#2196F3'] * len(tuned_results) + ['#FF5722']
bars = ax.barh(all_names, all_r2, color=bar_colors, edgecolor='white', alpha=0.85)
ax.axvline(x=0.85, color='green', linestyle='--', linewidth=2, label='Target (0.85)')
for bar, r2_val in zip(bars, all_r2):
    ax.text(max(bar.get_width() + 0.01, 0.02), bar.get_y() + bar.get_height()/2,
            f'{r2_val:.4f}', va='center', fontweight='bold', fontsize=9)
ax.set_title('Model R² Score Comparison', fontsize=12, fontweight='bold')
ax.set_xlabel('R² Score')
ax.legend(fontsize=9); ax.grid(alpha=0.3, axis='x')

# 2. Actual vs Predicted (time series)
ax = axes[0, 1]
ax.plot(dates_test, y_test, label='Actual', color='#2196F3', linewidth=2.5)
ax.plot(dates_test, y_pred_ensemble, label='Predicted (Ensemble)', color='#F44336', linewidth=2, linestyle='--')
ax.fill_between(dates_test, y_test, y_pred_ensemble, alpha=0.15, color='purple')
ax.set_title(f'Actual vs Predicted (R² = {r2_ens:.4f})', fontsize=12, fontweight='bold')
ax.set_xlabel('Date'); ax.set_ylabel('Price ($)')
ax.legend(fontsize=10); ax.grid(alpha=0.3)
ax.tick_params(axis='x', rotation=30)

# 3. Scatter plot
ax = axes[0, 2]
ax.scatter(y_test, y_pred_ensemble, alpha=0.6, c='#2196F3', edgecolors='white', s=60)
mn = min(y_test.min(), y_pred_ensemble.min())
mx = max(y_test.max(), y_pred_ensemble.max())
ax.plot([mn, mx], [mn, mx], 'r--', linewidth=2, label='Perfect Prediction')
ax.set_title(f'Scatter: Actual vs Predicted', fontsize=12, fontweight='bold')
ax.set_xlabel('Actual ($)'); ax.set_ylabel('Predicted ($)')
ax.legend(); ax.grid(alpha=0.3)

# 4. Residuals histogram
ax = axes[1, 0]
residuals = y_test - y_pred_ensemble
ax.hist(residuals, bins=20, color='#9C27B0', edgecolor='white', alpha=0.85)
ax.axvline(x=0, color='red', linestyle='--', linewidth=2)
ax.set_title('Residual Distribution', fontsize=12, fontweight='bold')
ax.set_xlabel('Residual (Actual - Predicted)'); ax.set_ylabel('Count')
ax.grid(alpha=0.3)

# 5. Feature importance (best model)
ax = axes[1, 1]
best_name = max(tuned_results, key=lambda k: tuned_results[k]['r2'])
best_model = tuned_results[best_name]['model']
if hasattr(best_model, 'feature_importances_'):
    imp = pd.Series(best_model.feature_importances_, index=feature_cols).sort_values(ascending=True)
    imp.tail(15).plot(kind='barh', ax=ax, color='#4CAF50', edgecolor='white', alpha=0.85)
    ax.set_title(f'Top 15 Features ({best_name})', fontsize=11, fontweight='bold')
    ax.set_xlabel('Importance')
    ax.grid(alpha=0.3, axis='x')

# 6. Residuals over time
ax = axes[1, 2]
ax.plot(dates_test, residuals, 'o-', color='#FF9800', alpha=0.7, markersize=5)
ax.axhline(y=0, color='red', linestyle='--', linewidth=2)
ax.fill_between(dates_test, residuals, 0, alpha=0.15, color='#FF9800')
ax.set_title('Residuals Over Time', fontsize=12, fontweight='bold')
ax.set_xlabel('Date'); ax.set_ylabel('Residual ($)')
ax.grid(alpha=0.3)
ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('stock_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()
print(" Evaluation plots saved!")

## Step 10: Predict Next 30 Days (Future Forecast)

In [ ]:
FORECAST_DAYS = 30

# Build a rolling forecast
# We need to reconstruct features for each future day
forecast_df = feat_df.copy()
last_date = df['Date'].max()

future_prices = []
future_dates_list = []

for i in range(FORECAST_DAYS):
    # Get the last row's features
    last_features = forecast_df[feature_cols].iloc[-1:].values
    last_features_s = scaler.transform(last_features)
    
    # Ensemble prediction
    pred = 0
    for name, res in tuned_results.items():
        pred += weights[name] * res['model'].predict(last_features_s)[0]
    
    future_prices.append(pred)
    next_date = last_date + pd.Timedelta(days=i + 1)
    # Skip weekends
    while next_date.dayofweek >= 5:
        next_date += pd.Timedelta(days=1)
    future_dates_list.append(next_date)
    
    # Append predicted row to forecast_df for next iteration's lag features
    new_row = forecast_df.iloc[-1:].copy()
    new_row['Date'] = next_date
    new_row[TARGET] = pred
    
    # Update lag features for next prediction
    for lag in [1, 2, 3, 5, 7, 10, 14]:
        col_name = f'{TARGET}_lag{lag}'
        if col_name in new_row.columns and len(forecast_df) >= lag:
            new_row[col_name] = forecast_df[TARGET].iloc[-lag]
    
    forecast_df = pd.concat([forecast_df, new_row], ignore_index=True)

future_prices = np.array(future_prices)
future_dates = np.array(future_dates_list)

# Plot
fig, ax = plt.subplots(figsize=(14, 6))

# Show last 60 days of actual + future forecast
ax.plot(df['Date'].tail(60), df[TARGET].tail(60), label='Historical Price', color='#2196F3', linewidth=2)
ax.plot(future_dates, future_prices, label=f'Forecast ({FORECAST_DAYS} days)',
        color='#FF6F00', linewidth=2.5, linestyle='--', marker='o', markersize=4)
ax.axvline(x=last_date, color='gray', linestyle=':', linewidth=1.5, label='Forecast Start')

# Confidence band
noise = rmse_ens  # Use RMSE as confidence width
ax.fill_between(future_dates, future_prices - noise, future_prices + noise,
                alpha=0.2, color='#FF6F00', label=f'±1 RMSE Confidence')

ax.set_title(f'{TARGET} -- {FORECAST_DAYS}-Day Ensemble Forecast', fontsize=14, fontweight='bold')
ax.set_xlabel('Date'); ax.set_ylabel('Price ($)')
ax.legend(fontsize=11); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('stock_forecast.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n 30-Day Forecast Summary:")
print(f"  Start price: ${future_prices[0]:.2f}")
print(f"  End price:   ${future_prices[-1]:.2f}")
print(f"  Change:      {((future_prices[-1]/future_prices[0])-1)*100:.2f}%")

In [ ]:
# ============ FINAL SUMMARY TABLE ============
print("\n" + "=" * 65)
print("              FINAL MODEL SUMMARY -- STOCK PREDICTION")
print("=" * 65)
print(f"  Approach          : Ensemble of {len(tuned_results)} ML models")
print(f"                      (NO LSTM -- only 365 data points, too few for DL)")
print(f"  Target            : {TARGET}")
print(f"  Features          : {len(feature_cols)} (lag, MA, EMA, RSI, MACD, BB, etc.)")
print(f"  Train Samples     : {len(X_train)}")
print(f"  Test Samples      : {len(X_test)}")
print(f"  -----------------------------------------")

for name, res in tuned_results.items():
    print(f"  {name:<28}: R²={res['r2']:.4f}")

print(f"  -----------------------------------------")
print(f"  ENSEMBLE R² Score    : {r2_ens:.4f}")
print(f"  ENSEMBLE RMSE        : ${rmse_ens:.4f}")
print(f"  ENSEMBLE MAE         : ${mae_ens:.4f}")
print(f"  ENSEMBLE MAPE        : {mape_ens:.2f}%")
print(f"  Directional Accuracy : {dir_acc:.2f}%")
print("=" * 65)

# Save model
import pickle
model_data = {
    'models': {name: res['model'] for name, res in tuned_results.items()},
    'weights': weights,
    'scaler': scaler,
    'features': feature_cols
}
with open('stock_model.pkl', 'wb') as f:
    pickle.dump(model_data, f)
print("\n Ensemble model saved to stock_model.pkl")